# Lab 07 — Root Cause Analysis

告警告訴你「有東西不對」；RCA 要回答「為什麼」。

這是 pipeline 裡最需要人類介入的步驟：
LLM 可以快速綜合多維度的 metric 證據、提出假說、建議處置——
但確認根因、決定要不要執行 runbook，仍然是人類的責任。

本 lab 涵蓋：
1. 為什麼 RCA 需要 LLM（rule-based 的局限）
2. 如何設計 RCA context 視窗（哪些資料要放進 prompt）
3. 把 Lab 05 的 reduced alerts 轉成 RCA candidates
4. 設計 prompt、呼叫 LLM、解析結構化輸出
5. 評估 LLM 診斷準確率（對照已知的合成事件標籤）
6. 生產 webhook 架構（alert → LLM → Slack / PagerDuty）


## Course architecture boundary

本課程有兩條資料路徑，請不要混在一起：

- **Actual operating path**: OS / network signals -> exporter -> Prometheus -> Grafana.
- **Python learning path**: organized telemetry CSV or previous notebook output -> Python analysis -> result CSV -> `outputs/prometheus-dropzone/current_results.csv` -> `python_results_exporter` -> Prometheus -> Grafana.

除非 cell 明確寫的是 operational deployment example，Python anomaly / forecasting / RCA notebooks 的主要輸入是 organized telemetry CSV 或前一個 notebook 的輸出，不是 Prometheus query。Grafana 也不直接讀 CSV；Grafana 查 Prometheus。


In [ ]:
from pathlib import Path
from IPython.display import SVG, display

def find_project_root(start=Path.cwd()):
    start = start.resolve()
    for base in (start, *start.parents):
        if (base / "environment.yml").exists():
            return base
    raise FileNotFoundError("Could not find project root containing environment.yml")

svg_candidates = [
    Path("diagrams/lab07_pipeline_position.svg"),
    find_project_root() / "labs/self-study" / "diagrams/lab07_pipeline_position.svg",
]
for svg_path in svg_candidates:
    if svg_path.exists():
        display(SVG(filename=str(svg_path)))
        break
else:
    raise FileNotFoundError("Could not find diagram: diagrams/lab07_pipeline_position.svg")


## Lab 07 概覽

### 學習目標

1. 理解症狀（symptom）vs 根因（root cause）的區別，以及 RCA Ladder 的推理框架
2. 建立結構化的 RCA rules，將 metric 證據轉成根因假說
3. 設計 LLM prompt：system prompt 角色、structured JSON 輸出、chain-of-thought 引導
4. 評估 LLM 診斷準確率並理解其失敗型態
5. 理解生產環境的 webhook 架構

### 前置條件

Lab 01、Lab 05 完成（features.csv、reduced_alerts.csv 存在）

## 設計主線：RCA 是證據組織，不是讓 LLM 猜答案

本章的實務連結是事件處置。告警只說「哪裡不正常」，RCA 要把 metric、拓樸、時間窗、歷史事件和變更紀錄整理成可驗證的假說。

設計 RCA 架構時請問三個問題：

1. **context 是否足夠但不過量**：LLM 需要摘要後的 evidence，不需要整段 raw time series。
2. **假說是否可驗證**：每個 root-cause hypothesis 都應該對應到下一個檢查動作，例如查 switch log、SFP power、STP 狀態。
3. **輸出是否可被系統接住**：JSON schema、confidence、recommended action、escalation flag 都要能進 ticket 或 webhook。

設計原則：LLM 不是根因裁判。它是把證據整理成候選解釋的助手，最後仍需要 human approval gate。


## 為什麼 RCA 需要 LLM？

### Rule-based RCA 的局限

傳統做法是手工維護一張 rule 表：

```python
if error_rate_high and discard_rate_high:
    return "Queue congestion"
if broadcast_zscore > 5:
    return "Broadcast storm"
```

問題：
- 每種新的失敗型態都需要有人寫新的 rule
- 組合爆炸：4 個 metric × 多個 port × 時間窗 → 規則數量難以維護
- 沒有解釋能力：只能說「是什麼」，不說「為什麼」
- 無法處理罕見或複合事件

### LLM RCA 的優勢

LLM 接收結構化的 metric 證據，用自然語言推理：

```
You are an AIOps assistant. The following is monitoring data for a network event:
  Event time: 2024-03-15 14:23 ~ 14:47
  Affected interface: edge-fw-01 / port-id7427 (firewall)
  traffic_total: average baseline 1.2M bytes, during event 8.7M bytes (+625%)
  error_rate: normal (z=0.3)
  broadcast_total: z=8.2; 3 other ports also showed z > 5 at the same time
  Recent changes: none

Analyze the likely root cause and provide: 1) problem type  2) key evidence  3) probable root cause  4) recommended action
```

LLM 可以：
- 整合跨 port 的同步異常型態
- 結合業務背景（port_role、設備類型）推理
- 對罕見組合給出有意義的假說
- 輸出可讀的診斷報告

**核心 insight：LLM 不是在「預測」根因，它是在「推理」——把你手邊的 metric 證據整理成一個有邏輯的假說。**


## RCA Context 視窗 設計

什麼資料放進 prompt，決定了診斷品質。這是最需要人類判斷的設計步驟。

### 必要資料（每個 alert 都要有）

| 類別 | 欄位 | 說明 |
|---|---|---|
| 時間 | start_time, end_time | 事件窗口 |
| 設備 | device_id, port_id, port_role | 拓樸位置 |
| 異常摘要 | problem_type, severity_score | Lab 05 的分類 |
| Metric 值 | 事件期間各 metric 的 mean, max, z-score | 量化證據 |
| Baseline | 1 小時前的 rolling mean | 對比用 |
| 跨 port | 同一時間其他 port 有無類似異常 | 廣播風暴 / 上游事件的關鍵線索 |

### 有用但非必要

| 類別 | 說明 |
|---|---|
| 最近變更 | 過去 24 小時有無設備變更、upgrade |
| 歷史事件 | 相同 port 過去 30 天有無類似事件 |
| 業務背景 | 這個 port 對應哪個應用 / 客戶 |

### Context 越豐富不代表越好

Token 是有成本的。Context 過長的問題：
- 重要資訊被稀釋（LLM attention 分散）
- API 成本上升
- 超過 context 視窗 限制

實務原則：**以最少的欄位呈現最強的信號**。
下面的 `build_rca_context()` 示範如何選取關鍵欄位。


In [ ]:
from pathlib import Path
import json
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

def show_fig(fig):
    display(fig)
    plt.close(fig)

PROJECT_ROOT = Path.cwd()
while PROJECT_ROOT != PROJECT_ROOT.parent:
    if (PROJECT_ROOT / "environment.yml").exists():
        break
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_PROCESSED = PROJECT_ROOT / "outputs" / "self-study"
REPORTS = PROJECT_ROOT / "reports"
REPORTS.mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")


In [ ]:
features = pd.read_csv(DATA_PROCESSED / "features.csv", parse_dates=["timestamp"])
reduced = pd.read_csv(DATA_PROCESSED / "reduced_alerts.csv", parse_dates=["start_time", "end_time"])
print(reduced.shape)
display(reduced.head())

---
📖 **概念說明 ▸ 講師引導** — 講師帶領說明約 8 分鐘，請先不要執行 cell。

---

## 📖 概念：症狀 vs 根因 — RCA Ladder

### 告警告訴你什麼，不告訴你什麼

告警系統（Lab 02–05）回答的是：「某個 metric 目前超出了正常範圍。」
這是**症狀（symptom）**，不是根因（root cause）。

```
Symptoms (observable):
    port-7427 error_rate elevated
    port-7427 discard_rate elevated
    port-7427 traffic_total decreased

Possible root causes (underlying cause):
    ├── Hardware: SFP module failure → high CRC errors → queue drop
    ├── Config: duplex mismatch → one-way transmission failure
    ├── External: upstream router path change → traffic suddenly rerouted
    └── Software: device firmware bug triggered by specific traffic pattern
```

同樣的症狀可能有完全不同的根因；不同的根因需要完全不同的修復行動。

### RCA Ladder：推理的層次

RCA Ladder 是一個從症狀到根因的推理框架：

```
Level 4: Business impact    "Customer A service disruption"
    ↑ inferred from
Level 3: System symptoms    "port-7427 error_rate elevated"  "port-7428 broadcast storm"
    ↑ inferred from
Level 2: Device behavior    "edge-fw-01 queue backlog started at 14:00"
    ↑ inferred from
Level 1: Root cause         "SFP module TX power dropped to -8 dBm (below -5 dBm spec)"
```

AIOps 系統最擅長 Level 3（症狀偵測）和 Level 2（設備行為關聯）。
Level 1（根本原因）和 Level 4（業務影響評估）仍然需要人類工程師。

LLM 在 Level 2 → Level 1 的推論特別有價值：
它可以快速綜合多個 port 的跨維度 metric 證據，提出假說。
但假說需要人類驗證——LLM 不能登入交換器確認 SFP 功率。

### RCA Rules 的角色

在 LLM 之前，先用 rule-based RCA 把症狀映射到候選根因。
這有兩個好處：
1. 快速、可靠——不需要 LLM 就能給出初步診斷
2. 結構化的輸出可以作為 LLM 的 evidence input，提高 LLM 推理品質

## Step 1 - 建立 RCA rules

把 reduced alert 的 evidence metrics 轉成 root cause candidate。

In [ ]:
def candidate_from_row(row):
    metrics = set(str(row.get("evidence_metrics", "")).split(","))
    problem = row.get("problem_type", "General anomaly")
    if "Queue congestion" in problem or "Queue / buffer" in problem:
        return "Queue congestion / buffer pressure", [
            "OCTETS high", "DISCARDS high", "ERRORS normal or not dominant",
            "Check QoS, queue, bandwidth, top talkers, scheduled backup jobs",
        ]
    if "Broadcast" in problem:
        return "Broadcast storm / L2 loop", [
            "BROADCAST high across ports", "NUCAST high may rise together",
            "Check STP, loop, ARP storm, DHCP storm, VLAN scope",
        ]
    if "Multicast" in problem:
        return "Multicast flooding", [
            "MULTICAST high across ports", "DISCARDS may rise",
            "Check IGMP snooping, querier, IPTV or routing protocol behavior",
        ]
    if "Link quality" in problem or "errors" in metrics:
        return "Link quality issue", [
            "ERRORS spike or repeated spikes", "Traffic may not be high",
            "Check cable, SFP, NIC, port, duplex mismatch",
        ]
    if "unknown_protocol" in metrics:
        return "Unknown protocol / scan", [
            "INUNKNOWNPROTOS high", "INPKTS high",
            "Check non-standard protocol, scan, attack, VLAN/routing config, new device rollout",
        ]
    if "Packet surge" in problem:
        return "Small packet scan or connection surge", [
            "UCASTPKTS high", "OCTETS may be flat or only mildly high",
            "Check port scan, short connections, DDoS small packets, heartbeat surge",
        ]
    if "Traffic surge" in problem:
        return "Business traffic growth or large transfer", [
            "OCTETS high", "Compare packet growth and avg_packet_size",
            "Check backup, data sync, download, upload, streaming, or capacity pressure",
        ]
    return "General network anomaly", ["Review metric trend, affected ports, recent changes"]

---
💻 **自行執行** — 閱讀說明並依序執行以下 cell。有疑問請舉手。

---

## Step 2 - 產生 RCA event table 與 scores

In [ ]:
rca_rows = []
for idx, row in reduced.iterrows():
    candidate, evidence = candidate_from_row(row)
    duration_minutes = max(5, int((row["end_time"] - row["start_time"]).total_seconds() / 60) + 5)
    score = min(100, int(row["severity_score"]) + 3 * len(evidence) + row["affected_port_count"] * 4)
    rca_rows.append(
        {
            "rca_event_id": f"RCA-{idx + 1:03d}",
            "start_time": row["start_time"],
            "end_time": row["end_time"],
            "affected_ports": row["affected_ports"],
            "problem_type": row["problem_type"],
            "root_cause_candidate": candidate,
            "evidence": " | ".join(evidence),
            "duration_minutes": duration_minutes,
            "severity_score": row["severity_score"],
            "root_cause_score": score,
            "confidence": "High" if score >= 85 else "Medium" if score >= 65 else "Low",
        }
    )

rca = pd.DataFrame(rca_rows).sort_values(["root_cause_score", "start_time"], ascending=[False, True])
display(rca.head(12))

---
💻 **自行執行** — 閱讀說明並依序執行以下 cell。有疑問請舉手。

---

## Step 3 - 視覺化：Root cause candidate ranking 與 confidence

In [ ]:
candidate_summary = (
    rca.groupby("root_cause_candidate")
    .agg(events=("rca_event_id", "count"), avg_score=("root_cause_score", "mean"), max_score=("root_cause_score", "max"))
    .sort_values("events", ascending=False)
)
display(candidate_summary)

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
candidate_summary["events"].sort_values().plot(kind="barh", ax=axes[0], color="tab:blue")
axes[0].set_title("RCA candidate event count")
axes[0].set_xlabel("events")

rca["confidence"].value_counts().reindex(["Low", "Medium", "High"]).fillna(0).plot(kind="bar", ax=axes[1], color="tab:green")
axes[1].set_title("Confidence distribution")
axes[1].set_ylabel("events")
plt.tight_layout()
show_fig(fig)

---
💻 **自行執行** — 閱讀說明並依序執行以下 cell。有疑問請舉手。

---

## Step 4 — 建立豐富的 RCA context（metric 歷史 + 跨 port）

在把 candidate 送給 LLM 之前，先為每個事件組裝完整的 metric 證據窗口。


In [ ]:
def build_rca_context(event_row, features_by_port):
    """Build a compact context dict for one RCA event."""
    t0, t1 = event_row["start_time"], event_row["end_time"]
    ports = str(event_row["affected_ports"]).split(",")
    primary_port = ports[0].strip()
    port_features = features_by_port.get(primary_port, pd.DataFrame())

    # Metric window: 30 min before event to end of event
    if not port_features.empty:
        window = port_features[
            (port_features["timestamp"] >= t0 - pd.Timedelta("30min"))
            & (port_features["timestamp"] <= t1)
        ].copy()
    else:
        window = pd.DataFrame()

    metric_cols = ["traffic_total", "error_rate", "discard_rate", "broadcast_total"]
    stats = {}
    if not window.empty:
        for col in metric_cols:
            if col in window.columns:
                pre = window[window["timestamp"] < t0][col]
                during = window[window["timestamp"] >= t0][col]
                stats[col] = {
                    "baseline_mean": round(float(pre.mean()), 2) if not pre.empty else None,
                    "event_mean": round(float(during.mean()), 2) if not during.empty else None,
                    "event_max": round(float(during.max()), 2) if not during.empty else None,
                    "z_score": round(
                        (float(during.mean()) - float(pre.mean())) / max(float(pre.std()), 1e-6), 1
                    ) if (not during.empty and not pre.empty) else None,
                }

    role = "unknown"
    if not port_features.empty and "port_role" in port_features.columns:
        role = str(port_features["port_role"].iloc[0])

    return {
        "event_id": event_row.get("rca_event_id", "?"),
        "event_start": t0,
        "event_end": t1,
        "time_window": f"{t0} to {t1}",
        "duration_minutes": round((t1 - t0).total_seconds() / 60),
        "primary_port": primary_port,
        "port_role": role,
        "device_id": event_row.get("device_id", "?"),
        "problem_type": event_row.get("problem_type", "?"),
        "rca_candidate": event_row.get("root_cause_candidate", "?"),
        "severity_score": event_row.get("severity_score", "?"),
        "metrics": stats,
        "total_affected_ports": event_row.get("affected_port_count", 1),
        "recent_changes": "None known",  # placeholder: connect to a change-management source in production
    }

# Build context for a bounded teaching sample.
# The reduced alert table may contain thousands of groups; RCA review normally starts with the highest-severity cases.
features_by_port = {
    str(port): group.sort_values("timestamp").reset_index(drop=True)
    for port, group in features.groupby("port_id", sort=False)
}
context_events = rca.sort_values("severity_score", ascending=False).head(25).copy()
rca_contexts = [build_rca_context(row, features_by_port) for _, row in context_events.iterrows()]

print(f"Built context for {len(rca_contexts)} RCA events from {len(rca)} reduced alerts")
print("\nSample context (highest-severity event):")
print(json.dumps(rca_contexts[0], indent=2, ensure_ascii=False, default=str))


---
📖 **概念說明 ▸ 講師引導** — 講師帶領說明約 7 分鐘，請先不要執行 cell。

---

## 📖 概念：為什麼要結構化 JSON 輸出？

### 自然語言輸出的問題

如果 LLM 的輸出是自由文字：

```
"Based on the anomalous rise in port-7427 error_rate and the cross-port synchronisation
  of broadcast_total, the most likely root cause is a broadcast storm;
  immediate inspection of STP state is recommended."
```

後端程式需要**解析**這段文字才能知道：
- 問題類型是什麼？（要寫入 incident ticket 的 `problem_type` 欄位）
- 是否需要升級？（決定是否 page on-call）
- 建議的行動是什麼？（觸發哪個 runbook）

解析自然語言既脆弱又昂貴——任何措辭的變化都可能讓 parser 失效。

### 結構化 JSON 的優勢

```json
{
  "problem_type": "broadcast_storm",
  "confidence": "High",
  "escalation_needed": true,
  "top_hypothesis": "STP loop between port-7427 and port-7428",
  "supporting_evidence": ["broadcast_total z=8.2 across 3 ports", "simultaneous onset at 14:03"],
  "recommended_actions": ["check spanning-tree status", "isolate port-7428"]
}
```

這個輸出可以：
- 直接存入資料庫（structured incident record）
- 按 `confidence` 排序（只給工程師看 Medium 以上的假說）
- 按 `escalation_needed` 觸發 PagerDuty
- 按 `recommended_actions` 執行 runbook automation

### 如何讓 LLM 穩定輸出 JSON？

1. **System prompt 明確規定**：「你的輸出必須是且只能是有效的 JSON，包含以下欄位…」
2. **提供 schema 範例**：在 prompt 裡給一個符合格式的例子（few-shot）
3. **Chain-of-thought 在 JSON 內**：讓 LLM 在 JSON 裡包含一個 `reasoning` 欄位，
   先寫推理過程，再得出結論（而不是讓 CoT 在 JSON 外面）
4. **Parser 驗證**：解析後驗證必要欄位是否存在，遺漏時觸發重試或降級

### Prompt 設計的三個要素

```
System prompt:  "You are a senior AIOps engineer … output JSON only … field definitions …"
    ↓
Evidence block: structured metric data (output of build_rca_context)
    ↓
CoT guidance:   "List evidence for and against each hypothesis, then select the most likely root cause"
```

Evidence block 的品質決定了 80% 的輸出品質。
LLM 無法從缺失的資料裡推論；你給它什麼，它用什麼。

## Step 5 — Prompt 設計

好的 RCA prompt 有三個要素：

1. **System prompt**：定義 LLM 的角色、輸出格式、限制條件
2. **Evidence**：結構化的 metric 資料（上面 build_rca_context 的輸出）
3. **Chain-of-thought 引導**：要求 LLM 先列舉證據、再下結論

輸出要求結構化 JSON，方便後續解析和存入資料庫。


In [ ]:
SYSTEM_PROMPT = """你是一位資深 AIOps 工程師，專長是網路設備的異常根因分析。

你會收到一個告警事件的監控資料，需要：
1. 分析各 metric 的異常型態
2. 結合跨 port 資訊判斷事件範圍
3. 提出可能根因（1-3 個假說，按可能性排序）
4. 給出具體的處置建議

**輸出必須是嚴格的 JSON 格式**，schema 如下：
{
  "problem_type": "一句話描述問題型態",
  "primary_evidence": ["主要證據 1", "主要證據 2", ...],
  "root_cause_hypotheses": [
    {"rank": 1, "hypothesis": "根因假說", "confidence": "High/Medium/Low", "reasoning": "推理過程"},
    ...
  ],
  "recommended_actions": ["立即行動 1", "立即行動 2", ...],
  "escalation_needed": true/false,
  "escalation_reason": "若需要 escalation 的原因（否則為 null）"
}

不要輸出 JSON 以外的任何文字。
"""

def build_prompt(context: dict) -> str:
    metrics_lines = []
    for metric, stat in context.get("metrics", {}).items():
        if stat.get("baseline_mean") is not None:
            metrics_lines.append(
                f"  {metric}: baseline={stat['baseline_mean']}, "
                f"event_mean={stat['event_mean']}, "
                f"event_max={stat['event_max']}, "
                f"z_score={stat['z_score']}"
            )
        else:
            metrics_lines.append(f"  {metric}: 資料不足")

    corr = context.get("correlated_ports", [])
    cross_info = f"同時異常的其他 port：{', '.join(corr[:5])}" if corr else "無跨 port 異常"

    return f"""網路告警事件摘要：

事件 ID：{context['event_id']}
事件Time：{context['time_window']}（持續 {context['duration_minutes']} 分鐘）
設備：{context['device_id']}
主要介面：{context['primary_port']}（角色：{context['port_role']}）
問題型態（Lab 05 分類）：{context['problem_type']}
影響介面數：{context['total_affected_ports']}
嚴重度分數：{context['severity_score']}
最近變更：{context['recent_changes']}

Metric 異常資料（事件窗口 vs 30 分鐘 baseline）：
{chr(10).join(metrics_lines)}

{cross_info}

請根據以上資料分析根因，輸出 JSON。"""

# Preview prompt for first event
sample_prompt = build_prompt(rca_contexts[0])
print("=== SYSTEM PROMPT ===")
print(SYSTEM_PROMPT[:300] + "...")
print()
print("=== USER PROMPT (first event) ===")
print(sample_prompt)


---
💻 **自行執行** — 閱讀說明並依序執行以下 cell。有疑問請舉手。

---

## Step 6 — 產生 RCA 診斷回應

課堂版本使用固定的 mock response，讓所有學員不需要註冊任何 AI 服務也能完整跑完流程。這一段的重點不是串接某一家模型，而是學會三件事：如何整理 evidence、如何要求結構化輸出、如何驗證診斷是否被資料支持。

若日後要接真實模型，請把 `call_llm()` 這個函式替換成自己的 provider adapter。可選方向很多，例如 OpenAI、Anthropic、Google Gemini、Azure OpenAI、AWS Bedrock、Cohere、Mistral，或 self-hosted 的 Ollama、vLLM、llama.cpp。課程不綁定任何一家。


In [ ]:
import time

MOCK_RESPONSE = {
    "problem_type": "Broadcast storm / L2 loop",
    "primary_evidence": [
        "broadcast_total z-score = 8.2（遠超threshold 3）",
        "同Time 3 個 port 出現類似廣播異常",
        "traffic_total 同步上升但 error_rate 維持正常"
    ],
    "root_cause_hypotheses": [
        {
            "rank": 1,
            "hypothesis": "交換器 L2 loop（STP 故障或誤配置）",
            "confidence": "High",
            "reasoning": "多 port 同步廣播激增是 L2 loop 的典型特徵；STP 若失效，廣播幀會無限循環"
        },
        {
            "rank": 2,
            "hypothesis": "ARP storm（大量 ARP request 短Time廣播）",
            "confidence": "Medium",
            "reasoning": "ARP storm 也會造成廣播激增，但通常局限在單一 VLAN；若跨多 port 則較符合 loop 特徵"
        }
    ],
    "recommended_actions": [
        "立即確認 STP topology：show spanning-tree",
        "檢查最近是否有 port 加入或 VLAN 變更",
        "若確認 loop：shutdown 疑似造成 loop 的 port",
        "啟用 PortFast + BPDU Guard 在接取層 port"
    ],
    "escalation_needed": True,
    "escalation_reason": "廣播風暴可能影響整個 VLAN segment，建議立即通知 NOC"
}


def call_llm(context: dict, use_mock: bool = True) -> dict:
    """Return a deterministic RCA response for class execution.

    To connect a real model after class, replace this function with a provider adapter
    that accepts the same context dict and returns the same JSON schema.
    """
    print(f"  [{context['event_id']}] Using course mock response")
    return MOCK_RESPONSE.copy()

# Run RCA for first 5 events (or all if small dataset)
batch = rca_contexts[:5]

print(f"Running RCA diagnosis for {len(batch)} events...\n")
llm_results = []
for ctx in batch:
    result = call_llm(ctx)
    llm_results.append({"event_id": ctx["event_id"], "context": ctx, "llm": result})
    print(f"✅ {ctx['event_id']} — {result.get('problem_type','?')}")
    time.sleep(0.1)

print(f"\nDone. {len(llm_results)} events diagnosed.")


---
💻 **自行執行** — 閱讀說明並依序執行以下 cell。有疑問請舉手。

---

## Step 7 — 解析結構化輸出

結構化 JSON 輸出讓你可以：
- 按 `confidence` 排序 hypotheses
- 提取 `recommended_actions` 觸發 runbook
- 按 `escalation_needed` 決定是否通知 NOC
- 把結果存入 incident management 系統


In [ ]:
def parse_llm_result(item: dict) -> dict:
    event_id = item["event_id"]
    llm = item["llm"]
    ctx = item["context"]

    top_hypothesis = llm.get("root_cause_hypotheses", [{}])[0]
    return {
        "event_id": event_id,
        "time_window": ctx["time_window"],
        "primary_port": ctx["primary_port"],
        "port_role": ctx["port_role"],
        "problem_type": llm.get("problem_type", "?"),
        "top_hypothesis": top_hypothesis.get("hypothesis", "?"),
        "top_confidence": top_hypothesis.get("confidence", "?"),
        "top_reasoning": top_hypothesis.get("reasoning", "?")[:120],
        "actions_count": len(llm.get("recommended_actions", [])),
        "first_action": (llm.get("recommended_actions") or ["?"])[0],
        "escalation_needed": llm.get("escalation_needed", False),
    }

rca_parsed = pd.DataFrame([parse_llm_result(r) for r in llm_results])
display(rca_parsed)

# Escalation summary
escalate = rca_parsed[rca_parsed["escalation_needed"] == True]
print(f"\n需要 escalation 的事件：{len(escalate)} / {len(rca_parsed)}")
if not escalate.empty:
    for _, row in escalate.iterrows():
        print(f"  🔴 {row['event_id']} — {row['problem_type']} ({row['top_confidence']})")
        print(f"     立即行動：{row['first_action']}")


---
💻 **自行執行** — 閱讀說明並依序執行以下 cell。有疑問請舉手。

---

## Step 8 — 評估 LLM 診斷準確率

本課程的合成資料有已知的事件標籤：
`traffic_surge`, `error_burst`, `discard_spike`, `traffic_drop`, `broadcast_storm`

把 LLM 的 `problem_type` 對照已知標籤，可以量化診斷準確率。
在真實環境裡，這對應的是「有 root cause 記錄的歷史 incident」作為 ground truth。


In [ ]:
# Known event type → expected LLM problem_type keywords
KNOWN_LABELS = {
    "traffic_surge": ["traffic", "surge", "Traffic", "激增"],
    "error_burst":   ["error", "錯誤", "CRC", "duplex"],
    "discard_spike": ["discard", "queue", "丟棄", "buffer", "congestion"],
    "traffic_drop":  ["drop", "下降", "link", "failure"],
    "broadcast_storm": ["broadcast", "storm", "loop", "廣播"],
}

def match_label(llm_problem_type: str, known_label: str) -> bool:
    keywords = KNOWN_LABELS.get(known_label, [])
    text = llm_problem_type.lower()
    return any(kw.lower() in text for kw in keywords)

# Load ground truth from features (event_type column if exists)
ground_truth_col = "event_type"
eval_rows = []

for item in llm_results:
    ctx = item["context"]
    event_id = ctx["event_id"]

    # Try to get ground truth from features during event window
    t0 = pd.Timestamp(ctx["event_start"])
    t1 = pd.Timestamp(ctx["event_end"])
    window_mask = (
        (features["timestamp"] >= t0) & (features["timestamp"] <= t1)
        & (features["port_id"] == ctx["primary_port"])
    )
    window_rows = features[window_mask]

    if ground_truth_col in features.columns and not window_rows.empty:
        gt_label = window_rows[ground_truth_col].mode().iloc[0]
    else:
        gt_label = "unknown"

    llm_problem = item["llm"].get("problem_type", "")
    correct = match_label(llm_problem, gt_label) if gt_label != "unknown" else None
    eval_rows.append({
        "event_id": event_id,
        "ground_truth": gt_label,
        "llm_diagnosis": llm_problem[:60],
        "match": correct,
    })

eval_df = pd.DataFrame(eval_rows)
display(eval_df)

known_mask = eval_df["ground_truth"] != "unknown"
if known_mask.sum() > 0:
    accuracy = eval_df.loc[known_mask, "match"].mean()
    print(f"\nDiagnosis accuracy (known labels only): {accuracy:.0%}  ({known_mask.sum()} events)")
else:
    print("\n合成資料中沒有 event_type 欄位可供評估（正常）。")
    print("實際使用時，以歷史 incident records 作為 ground truth。")


---
💬 **討論暫停 ▸ 全班討論** — 停止執行，分享你的觀察。

---

## ⚠ 人類驗證點 #7 — RCA 診斷是假說，不是事實

LLM 的診斷是根據 metric 數據推論出的**假說**。
即使 confidence = "High"，也必須由人類確認後再執行 runbook。

| LLM 輸出 | 人類需要驗證的事 |
|---|---|
| `problem_type: Broadcast storm` | 實際登入交換器確認 `show spanning-tree` |
| `recommended_actions: shutdown port` | 確認 shutdown 的影響範圍，是否有 critical 服務在上面 |
| `escalation_needed: true` | 判斷業務衝擊等級，決定叫醒哪個 on-call |
| `top_confidence: Medium` | Medium 以下的假說需要更多證據再行動 |

### LLM RCA 的常見失敗型態

1. **Hallucination**：LLM 引用了不存在的 metric 或做了錯誤的數學
   → 永遠對照原始數字，不要只看 LLM 的 reasoning 文字

2. **Overconfidence**：LLM 對只有一個弱信號的情況也輸出 High confidence
   → 設定 business rule：單一 metric 異常的事件，無論 LLM 說什麼，confidence 降一級

3. **Context 不足**：LLM 沒有看到關鍵的跨 port 或拓樸資訊
   → 這是 context 視窗 設計的問題，回到 Step 2b 補充欄位

4. **語言 / 術語偏差**：LLM 輸出的 `problem_type` 和你的 ticketing system 分類不一致
   → 在 system prompt 裡列出允許的分類清單，要求 LLM 從中選擇


---
🔧 **探索練習** — 修改指定參數，觀察結果變化。

---

## 🔧 探索練習：修改 System Prompt 輸出格式

在 Step 5 的 `SYSTEM_PROMPT` 中，找到 `confidence` 的定義，將 `"High"/"Medium"/"Low"` 三級改成加入 `"Very Low"` 四級，並修改 parser（Step 7）讓它能處理新的值。

重新執行 Step 6，觀察 mock response 是否需要跟著 schema 調整。若你日後接入真實模型，也可以用同一份 schema 比較不同 provider 的輸出穩定性。

**如何判斷輸出是否合理**

- 有清楚 metric 佐證的事件（廣播風暴：3 個 port 同時 broadcast z > 5）→ 應該是 `High`
- 只有一個弱信號的事件（traffic_total z = 2.1）→ 應該是 `Low` 或 `Very Low`
- 若模型對只有一個弱信號的事件輸出 `High`，表示 prompt 沒有充分說明 confidence 的定義

**讓你重新考慮的信號**

- 所有事件都輸出 `High`：confidence 定義不夠清晰
- 輸出了不在 schema 裡的值（如 `"high"` 小寫）：parser 的防禦性驗證需要加強

---

> 「如果診斷輸出說 confidence 是 High，但你看原始 metric 數字覺得不確定，你信誰？」

> 「build_rca_context 把跨 port 的資訊放進 prompt。如果你有 500 個 port，你不可能把所有 port 的資料都放進去。你會選哪些 port？」

> 「如果模型診斷準確率是 70%，但 rule-based RCA 的準確率是 80%，你還需要模型嗎？在什麼條件下它是必要的？」


## Step 9 — 生產 Webhook 架構（rca_webhook.py）

在生產環境，RCA 不是手動執行 lab，而是 alert 觸發自動流程：

```
Alertmanager
    │  webhook (HTTP POST)
    ▼
rca_webhook.py (Python HTTP server)
    │  pull metric context from Prometheus API in production
    │  or read organized CSV during course validation
    │  build_rca_context()
    │  diagnose_incident()
    │  parse_diagnosis_result()
    ▼
    ├─ ChatOps: human-readable diagnostic summary + recommended actions
    ├─ incident management: if escalation_needed=True, trigger incident
    └─ CMDB / ticketing: write RCA record for post-incident review
```

下面是最小可運行的 webhook server skeleton。請注意：課程內的 RCA 驗證主要使用整理好的 CSV 與前面 lab 輸出；Prometheus API pull 是生產部署時取得最新 metric context 的方式。它保留 provider adapter 的位置，但課程不綁定任何模型服務。


In [ ]:
# rca_webhook.py — 這段 code 展示 webhook 結構，不在 lab 裡執行
# 實際部署：python rca_webhook.py  （在 exporter 旁邊執行）

webhook_lines = [
    "from http.server import HTTPServer, BaseHTTPRequestHandler",
    "import json, os, threading",
    "import urllib.request, urllib.parse",
    "",
    "PROM_URL = 'http://localhost:9090'",
    "CHATOPS_WEBHOOK = os.environ.get('CHATOPS_WEBHOOK_URL', '')",
    "",
    "def pull_context_from_prometheus(alert):",
    "    port_id = alert['labels'].get('port_id', 'unknown')",
    "    return {'event_id': alert['labels'].get('alertname', '?'), 'primary_port': port_id}",
    "",
    "def diagnose_incident(ctx):",
    "    # Provider adapter boundary: class version calls mock response.",
    "    # Production may call any approved provider or local model.",
    "    return call_llm(ctx)",
    "",
    "class RCAHandler(BaseHTTPRequestHandler):",
    "    def do_POST(self):",
    "        body = self.rfile.read(int(self.headers['Content-Length']))",
    "        payload = json.loads(body)",
    "        for alert in payload.get('alerts', []):",
    "            threading.Thread(target=self.handle_alert, args=(alert,)).start()",
    "        self.send_response(200)",
    "        self.end_headers()",
    "",
    "    def handle_alert(self, alert):",
    "        ctx = pull_context_from_prometheus(alert)",
    "        diagnosis = diagnose_incident(ctx)",
    "        parsed = parse_llm_result({'event_id': ctx['event_id'], 'context': ctx, 'llm': diagnosis})",
    "",
    "if __name__ == '__main__':",
    "    server = HTTPServer(('0.0.0.0', 9095), RCAHandler)",
    "    server.serve_forever()",
]
WEBHOOK_SKELETON = "\n".join(webhook_lines)

print("=== rca_webhook.py skeleton ===")
print(WEBHOOK_SKELETON)
print("\nAlertmanager receiver example:")
print("  receivers:")
print("    - name: rca-webhook")
print("      webhook_configs:")
print("        - url: http://localhost:9095")


## 整合回顧：Lab 07 在 Pipeline 裡的位置

```
What you did in Lab 07               Production equivalent
────────────────────────            ─────────────────────────────────
build_rca_context()           →     CSV context in course; Prometheus API context in production
build_prompt()                →     prompt template (version-controlled)
call_llm()                    →     provider adapter or local model adapter
parse_llm_result()            →     structured output parser
evaluation (Step 6)           →     precision tracking dashboard (Grafana panel)
rca_webhook.py (Step 7)       →     production webhook service

⚠ Human verification point #7 →     NOC engineer reviews diagnosis before executing runbook
```

**Lab 07 輸出的東西**：
- `rca_lib.py` — 可複用的 context builder + diagnosis adapter（從 lab 匯出）
- `rca_webhook.py` — webhook server（直接部署，不需要重寫）
- 評估指標 — accuracy、confidence 分佈（監控診斷品質隨Time的變化）


## Optional: update Grafana with RCA-related numbers

RCA 的長文字 explanation 不適合直接放進 Prometheus。若要在 Grafana 顯示 RCA 周邊結果，請只掛數值欄位，例如 severity score、alert count、confidence score 或 escalation flag。

做法仍是：把整理好的 CSV 複製到 `outputs/prometheus-dropzone/current_results.csv`，並保持 `python infra/python_results_exporter.py` 執行。完整流程見 `labs/getting-started/05-prometheus-dropzone.md`。
